<a href="https://colab.research.google.com/github/MANI-WEBDEVE/RAG_SYSTEM/blob/main/RAG_CHUNK_CONCEPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

if "COLAB_GPU" in os.environ:
  print('ok')
  # !pip install -U torch # requires torch 2.1.1+ (for efficient sdpa implementation)
  !pip install PyMuPDF
  !pip install sentence-transformers # for embedding models
  !pip install tqdm
  !pip install accelerate
  !pip install bitsandbytes
  !pip install flash-attn --no-build-isolation

ok
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 84.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for flash-attn
  Running setup.py clean for flash-attn
Failed to build flash-attn
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (flash-attn)


In [2]:
!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U transformers sentence-transformers

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: sentence-transformers 5.3.0
Uninstalling sentence-transformers-5.3.0:
  Successfully uninstalled sentence-transformers-5.3.0
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 75.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 111.2 MB/s eta 0:00:00
     ━

## Download the Docs

In [3]:
import os
import requests

pdf_path = "ML.pdf"

# Download pdf if doesn`t exits
if not os.path.exists(pdf_path):
  print("File does not exist")

  # The URL of the PDF of Download
  url = 'https://shashwatwork.github.io/assets/files/ml_ebook.pdf'

  filename=pdf_path

  # send the GET request to download the PDF

  response = requests.get(url)

  # check if the request was successful
  if response.status_code == 200:
    # open a file in binary write mode and save the contant it
    with open(filename, 'wb') as file:
      file.write(response.content)
    print(f'write the file has been downloaded and saved as {filename}')
  else:
    print(f'the file failed to download status code: {response.status_code}')
else:
  print(f"file {pdf_path} exists")


File does not exist
write the file has been downloaded and saved as ML.pdf


## Extract the text from documents

In [4]:
from tqdm.auto import tqdm
import fitz

def text_formater(text:str) -> str:
  """ Perform the minner formatting """
  cleaned_text=text.replace("\n", " ").strip()

  return cleaned_text
# Note: focus on only text rather than images and table

def open_and_read_pdf(pdf_path:str)->list[dict]:
  """
  Open : the pdf and read the text page by page

  Parameter get function:
    pdf_path the file path your pdf to open and read the document
  Return:
    list of and dictionary fromat [{}, {}] to return this function

  """
  docs = fitz.open(pdf_path)
  page_and_text=[]

  for page_number , page in tqdm(enumerate(docs)):
    text=page.get_text()
    text=text_formater(text)
    page_and_text.append({
        "page_number": page_number,
        "page_total_words": len(text.split(' ')),
        "page_total_char": len(text),
        "page_total_sentence": len(text.split('. ')),
        "page_total_token": len(text) /4,
        "text": text
    })

  return page_and_text


page_and_text = open_and_read_pdf(pdf_path=pdf_path)
page_and_text[:20]




0it [00:00, ?it/s]

[{'page_number': 0,
  'page_total_words': 1,
  'page_total_char': 0,
  'page_total_sentence': 1,
  'page_total_token': 0.0,
  'text': ''},
 {'page_number': 1,
  'page_total_words': 1,
  'page_total_char': 0,
  'page_total_sentence': 1,
  'page_total_token': 0.0,
  'text': ''},
 {'page_number': 2,
  'page_total_words': 30,
  'page_total_char': 237,
  'page_total_sentence': 1,
  'page_total_token': 59.25,
  'text': 'Aurélien Géron Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow Concepts, Tools, and Techniques to Build Intelligent Systems SECOND EDITION Boston Farnham Sebastopol Tokyo Beijing Boston Farnham Sebastopol Tokyo Beijing'},
 {'page_number': 3,
  'page_total_words': 249,
  'page_total_char': 1777,
  'page_total_sentence': 14,
  'page_total_token': 444.25,
  'text': '978-1-492-03264-9 [LSI] Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow by Aurélien Géron Copyright © 2019 O’Reilly Media. All rights reserved. Printed in the United States of Am

In [5]:
import random
random_Sam= random.sample(page_and_text, k=5)

In [6]:
random_Sam

[{'page_number': 259,
  'page_total_words': 338,
  'page_total_char': 2124,
  'page_total_sentence': 15,
  'page_total_token': 531.0,
  'text': '>>> np.mean(y_train_partially_propagated == y_train[partially_propagated]) 0.9896907216494846 Active Learning To continue improving your model and your training set, the next step could be to do a few rounds of active learning: this is when a human expert interacts with the learn‐ ing algorithm, providing labels when the algorithm needs them. There are many dif‐ ferent strategies for active learning, but one of the most common ones is called uncertainty sampling: • The model is trained on the labeled instances gathered so far, and this model is used to make predictions on all the unlabeled instances. • The instances for which the model is most uncertain (i.e., when its estimated probability is lowest) must be labeled by the expert. • Then you just iterate this process again and again, until the performance improvement stops being worth the lab

In [7]:
import pandas as pd

df=pd.DataFrame(page_and_text)

In [8]:
df

,page_number,page_total_words,page_total_char,page_total_sentence,page_total_token,text
0,0,1,0,1,0.00,
1,1,1,0,1,0.00,
2,2,30,237,1,59.25,Aurélien Géron Hands-on Machine Learning with ...
3,3,249,1777,14,444.25,978-1-492-03264-9 [LSI] Hands-on Machine Learn...
4,4,2467,3246,90,811.50,Table of Contents 1. The Machine Learning Land...
...,...,...,...,...,...,...
274,274,190,1131,10,282.75,Figure 9-22. Bayesian Gaussian mixture model P...
275,275,350,1771,13,442.75,Bayes’ theorem (Equation 9-2) tells us how to ...
276,276,363,2263,15,565.75,"In practice, there are different techniques to..."
277,277,440,2785,19,696.25,Other Anomaly Detection and Novelty Detection ...


In [9]:
df.describe()

,page_number,page_total_words,page_total_char,page_total_sentence,page_total_token
count,279.000000,279.000000,279.000000,279.000000,279.000000
mean,139.000000,335.738351,1750.713262,13.591398,437.678315
std,80.684571,408.443051,691.596170,12.272303,172.899043
min,0.000000,1.000000,0.000000,1.000000,0.000000
25%,69.500000,222.000000,1367.500000,9.000000,341.875000
50%,139.000000,302.000000,1786.000000,12.000000,446.500000
75%,208.500000,356.500000,2124.500000,15.000000,531.125000
max,278.000000,3948.000000,4918.000000,124.000000,1229.500000


## Step no 3 Chunking Section
---
How to implement five methods of chunking. Fixed , Semantic , structural , recursive, LLM based.

## First Method is Fixed size of Chunk our data

In [10]:
def chunk_size(text:str, chunk_size:int = 500) -> list:
  """
  Split the text to chunk size.
  """

  chunks=[]
  words=text.split()
  current_chunk=""

  for word in words:
    # if check the word before adding excced chunk
    if len(current_chunk) + len(word) + 2 <= chunk_size:
      current_chunk += (word + " ")
    else:
      chunks.append(current_chunk.strip())
      current_chunk += word + " "

  if current_chunk:
      chunks.append(current_chunk.strip())

  return chunks


In [11]:
chunk_ex = []
current_chunk_ex = ""

ex_word = "this is my car"

In [12]:
for word in ex_word.split():
  if len(current_chunk_ex) + len(word) + 1 <= 3:
    print(word)
    current_chunk_ex += (word + " ")
    print(word + " ")
  else :
    print('lo')

lo
is
is 
lo
lo


In [13]:
current_chunk_ex

'is '

In [14]:
chunk_ex.append(ex_word.split())

In [15]:
chunk_ex

[['this', 'is', 'my', 'car']]

## Method Semantic Chunking

In [16]:
!pip install -q --upgrade "sentence-transformers==3.0.1" "transformers<5, >4.41" scikit-learn nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 164.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.6 MB/s eta 0:00:00


In [17]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk
nltk.download('punkt_tab', quiet=True)

# load model locally

semantic_model=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def semantic_chunk_text(text:str, similarity_threshold=0.8, max_token=500)-> list:

  sentence = nltk.sent_tokenize(text)

  if not sentence:
    return []

  embeddings=semantic_model.encode(sentence)

  chunks=[]
  current_chunk=[sentence[0]]
  current_embedding=embeddings[0]

  for i in range(1, len(sentence)):
    sim=cosine_similarity([current_embedding], [embeddings[i]])[0][0]
    chunk_token_count=len(" ".join(current_chunk)) // 4

    if sim >= similarity_threshold and chunk_token_count < max_token:
      current_chunk.append(sentence[i])
      current_embedding = np.mean([current_embedding, embeddings[i]], axis=0)
    else:
      chunks.append(" ".join(current_chunk))
      current_chunk = [sentence[i]]
      current_embedding=embeddings[i]

  if current_chunk:
    chunks.append(" ".join(current_chunk))
  return chunks

from tqdm.auto import tqdm

def semantic_chunk_pdf_pages(page_and_text:list, similarity_threshod:float=0.8, max_token:int=500)-> list[dict]:
    all_chunks=[]

    for page in tqdm(page_and_text, desc="semantic chunking pages"):
      page_number=page['page_number']
      page_text=page['text']

      chunks= semantic_chunk_text(
          page_text,
          similarity_threshold=similarity_threshod,
          max_token=max_token
      )

      for i , chunk in enumerate(chunks):
        all_chunks.append(
            {
                'page_number':page_number,
                'chunk_index': i,
                'chunk_char_count': len(chunk),
                'chunk_word_count': len(chunk.split()),
                'chunk_token_count': len(chunk) / 4,
                'chunk_text': chunk
            }
        )
    return all_chunks
import nltk
nltk.download('punkt_tab', quiet=True)
semactic_chunks = semantic_chunk_pdf_pages(page_and_text, similarity_threshod=0.8, max_token=500)





/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

semantic chunking pages:   0%|          | 0/279 [00:00<?, ?it/s]

In [18]:
len(semactic_chunks)

3461

In [19]:
total_chunk=0
for i in semactic_chunks:
  print(i)


{'page_number': 2, 'chunk_index': 0, 'chunk_char_count': 237, 'chunk_word_count': 30, 'chunk_token_count': 59.25, 'chunk_text': 'Aurélien Géron Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow Concepts, Tools, and Techniques to Build Intelligent Systems SECOND EDITION Boston Farnham Sebastopol Tokyo Beijing Boston Farnham Sebastopol Tokyo Beijing'}
{'page_number': 3, 'chunk_index': 0, 'chunk_char_count': 141, 'chunk_word_count': 18, 'chunk_token_count': 35.25, 'chunk_text': '978-1-492-03264-9 [LSI] Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow by Aurélien Géron Copyright © 2019 O’Reilly Media.'}
{'page_number': 3, 'chunk_index': 1, 'chunk_char_count': 20, 'chunk_word_count': 3, 'chunk_token_count': 5.0, 'chunk_text': 'All rights reserved.'}
{'page_number': 3, 'chunk_index': 2, 'chunk_char_count': 40, 'chunk_word_count': 7, 'chunk_token_count': 10.0, 'chunk_text': 'Printed in the United States of America.'}
{'page_number': 3, 'chunk_index': 3, 'chu

In [20]:
se=nltk.sent_tokenize(page_and_text[3]['text'])

In [21]:
semantic_model.encode(se)

array([[-0.08356416, -0.10359119,  0.05468197, ...,  0.01103478,
        -0.07604165,  0.01905698],
       [ 0.0043254 ,  0.03656445, -0.06567881, ...,  0.02435147,
        -0.05985817,  0.02358041],
       [-0.02225668,  0.06076472, -0.10556878, ..., -0.0474557 ,
         0.0873454 ,  0.01436314],
       ...,
       [-0.08282682,  0.12455787,  0.04454409, ..., -0.01990742,
         0.03544137, -0.05798681],
       [-0.04362151,  0.07473563, -0.05297586, ...,  0.03542088,
         0.02321107, -0.04479911],
       [-0.10693968,  0.00704439, -0.08952145, ...,  0.06513036,
         0.05043962, -0.09322445]], dtype=float32)

In [22]:
page_and_text[23]

{'page_number': 23,
 'page_total_words': 314,
 'page_total_char': 1757,
 'page_total_sentence': 14,
 'page_total_token': 439.25,
 'text': 'batch learning system can adapt to change. Simply update the data and train a new version of the system from scratch as often as needed. This solution is simple and often works fine, but training using the full set of data can take many hours, so you would typically train a new system only every 24 hours or even just weekly. If your system needs to adapt to rapidly changing data (e.g., to pre‐ dict stock prices), then you need a more reactive solution. Also, training on the full set of data requires a lot of computing resources (CPU, memory space, disk space, disk I/O, network I/O, etc.). If you have a lot of data and you automate your system to train from scratch every day, it will end up costing you a lot of money. If the amount of data is huge, it may even be impossible to use a batch learning algorithm. Finally, if your system needs to be able t

## Sturctural Chunking

In [54]:
import random
import textwrap
import re
# 1) Helper to detect "chapter start" pages
def _is_chapter_header_page(text: str) -> bool:
    """Detects chapter headers like 'Chapter 1:', 'Chapter 2:' etc."""
    return re.search(r"chapter\s+\d+", text, flags=re.IGNORECASE) is not None

def _guess_title_from_page(text: str) -> str:
    """Extracts chapter title from page text"""
    # Option 1: If format is "Chapter X: Title"
    m = re.search(r"chapter\s+\d+:\s*(.+)", text, flags=re.IGNORECASE)
    if m:
        return m.group(1).strip()

    # Option 2: First line as title
    lines = text.strip().split('\n')
    if lines:
        return lines[0].strip()

    # Fallback
    return "Untitled chapter"

In [55]:
# 2) Build chapter chunks
def chapter_chunk_pdf_pages(pages_and_texts: list[dict]) -> list[dict]:
    """
    Returns a list of chapter chunks:
    [
        {
            'chapter_index': int,
            'title': str,
            'page_start': int,  # adjusted page number (your -41 offset)
            'page_end': int,
            'chunk_char_count': int,
            'chunk_word_count': int,
            '# ~chars/4
            'chunk_token_count': float,  # round(len(all_text) / 4, 2),
            'chunk_text': str
        },
        ...
    ]
    """
    if not pages_and_texts:
        return []

    # Treat entire doc as one "chapter" if no headers found
    all_text = " ".join(p["text"] for p in pages_and_texts).strip()
    if not _is_chapter_header_page(all_text):
        return [{
            "chapter_index": 0,
            "title": _guess_title_from_page(pages_and_texts[0]["text"]),
            "page_start": pages_and_texts[0]["page_number"],
            "page_end": pages_and_texts[-1]["page_number"],
            "chunk_char_count": len(all_text),
            "chunk_word_count": len(all_text.split()),
            "chunk_token_count": round(len(all_text) / 4, 2),
            "chunk_text": all_text
        }]

    # Find all page indices that look like the start of a chapter
    chapter_starts = []
    for i, p in enumerate(pages_and_texts):
        txt = p["text"].strip()
        if _is_chapter_header_page(txt):
            chapter_starts.append(i)

    # Build chapter ranges (start -> next_start-1)
    chapter_chunks = []
    for ci, s in enumerate(chapter_starts):
        e = chapter_starts[ci + 1] - 1 if (ci + 1 < len(chapter_starts)) else (len(pages_and_texts) - 1)
        if e < s:  # guard (shouldn't happen)
            continue

        pages = pages_and_texts[s:e + 1]
        text_concat = " ".join(p["text"] for p in pages).strip()
        title = _guess_title_from_page(pages[0]["text"])

        chapter_chunks.append({
            "chapter_index": ci,
            "title": title,
            "page_start": pages[0]["page_number"],
            "page_end": pages[-1]["page_number"],
            "chunk_char_count": len(text_concat),
            "chunk_word_count": len(text_concat.split()),
            "chunk_token_count": round(len(text_concat) / 4, 2),
            "chunk_text": text_concat
        })

    return chapter_chunks

In [53]:
len(structural_chunks)

3

In [56]:
structure_chunked_pages = chapter_chunk_pdf_pages(page_and_text)

print(f"Total chapter-based chunks: {len(structure_chunked_pages)}")

if structure_chunked_pages:
    first = structure_chunked_pages[120]
    print(f"First chapter pages ({first['page_start']}-{first['page_end']}): {first['title']}")
    print(f"  {first['chunk_text'][:]} + \"...\"")
else:
    print("No chapters detected.")

Total chapter-based chunks: 158
First chapter pages (211-212): Ensemble Learning and Random Forests
  Figure 7-10. GBRT ensembles with not enough predictors (left) and too many (right) In order to find the optimal number of trees, you can use early stopping (see Chap‐ ter 4). A simple way to implement this is to use the staged_predict() method: it returns an iterator over the predictions made by the ensemble at each stage of train‐ ing (with one tree, two trees, etc.). The following code trains a GBRT ensemble with 120 trees, then measures the validation error at each stage of training to find the opti‐ mal number of trees, and finally trains another GBRT ensemble using the optimal number of trees: import numpy as np from sklearn.model_selection import train_test_split from sklearn.metrics import mean_squared_error X_train, X_val, y_train, y_val = train_test_split(X, y) gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=120) gbrt.fit(X_train, y_train) errors = [mean_squared_err